3장 코드를 하나하나 보기 전에 이것부터 기억하세요.

Transformer 모델을 사용하는 과정은 항상 다음 흐름입니다.

사람이 작성한 문장 -> Tokenizer -> input_ids / attention_mask -> Transformer Model -> logits / embedding -> 분류, 생성, 질의응답 등의 결과

예를 들어 사람이 "삼성전자가 새로운 스마트폰을 출시했다."라고 입력했다고 합시다.

컴퓨터는 이 문장을 그대로 이해하지 못합니다.

그래서 먼저 Tokenizer가

"삼성", "전자", "가", "새로운", "스마트폰", ...

같은 토큰으로 나눕니다.

그리고 실제 모델에는 문자열도 넣지 않습니다.

삼성 -> 11231

전자 -> 2345

가 -> 98359

처럼 숫자로 바꿉니다.

이것이 input_ids 입니다.

그리고 모델이

input_ids -> Embedding -> Transformer -> 분류 Head -> logits

과정을 거칩니다. 3장은 사실상 이 전체 과정을 Hugging Face로 구현해 보는 장입니다.



In [3]:
!pip install transformers==4.50.0 datasets==3.5.0 huggingface_hub==0.29.0 -qqq

Jupyter Notebook이나 Google Colab에서 "!명령어"라고 작성하면 Python 코드가 아니라 운영체제의 Shell 명령어를 실행합니다.

그래서 !pip install은 Python 문법이 아닙니다.

터미널에서 pip install ... 을 실행하는 것과 같습니다.

설치하는 것은 3개입니다.

transformers: BERT, RoBERTa, GPT 등 Transformer 모델

datasets: 머신러닝 데이터셋 다운로드/관리

huggingface_hub: Hugging Face Hub 다운로드/업로드

쉽게 비유하자면

transformers -> 모델 도구상자

datasets -> 데이터 도구상자

huggingface_hub -> 모델 저장소 연결 도구

입니다.

Python==4.50.0 은 버전을 고정합니다.

라이브러리는 계속 업데이트되기 때문에 버전 차이에 의해서 에러가 나는 것을 방지하기 위한 것입니다.

-qqq는 설치 로그를 최대한 조용하게 출력하라는 옵션입니다.



# 3.1절 허깅페이스 트랜스포머란?

In [4]:
from transformers import AutoTokenizer, AutoModel

text = "What is Huggingface Transformers?"
# BERT 모델 활용
bert_model = AutoModel.from_pretrained("bert-base-uncased")
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
encoded_input = bert_tokenizer(text, return_tensors='pt')
bert_output = bert_model(**encoded_input)
# GPT-2 모델 활용
gpt_model = AutoModel.from_pretrained('gpt2')
gpt_tokenizer = AutoTokenizer.from_pretrained('gpt2')
encoded_input = gpt_tokenizer(text, return_tensors='pt')
gpt_output = gpt_model(**encoded_input)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

AutoTokenizer와 AutoModel

from transformers import AutoTokenizer, AutoModel

두 클래스를 가져옵니다.

AutoTokenizer: 텍스트를 모델이 이해할 수 있는 숫자로 바꿉니다.

문장 -> token -> token ID

AutoModel: Transformer 모델 자체를 불러옵니다. 여기서 Auto가 굉장히 중요합니다. Hugging Face에서는 모델마다 별도의 클래스를 직접 사용할 수도 있습니다.

예를 들어 BertModel, GPT2Model, RobertaModel 등이 있습니다. 그런데 AutoModel을 사용하면

AutoModel.from_pretrained("bert-base-uncased")

라고만 해도 Hugging Face가 "아, 이건 BERT 구나."라고 알아서 적절한 클래스를 선택합니다.

즉, AutoModel -> 모델 ID 확인 -> BERT? GPT? RoBERTa? -> 자동 선택 입니다.

3. from_pretrained()가 굉장히 중요합니다.

bert_model = AutoModel.from_pretrained("bert-base-uncased")

LLM 공부하면서 엄청 자주 보게 될 코드입니다.

pretrained는 이미 학습되어 있는 이라는 뜻입니다. 즉, from_pretrained(...)는 Hugging Face Hub에 저장되어 있는 이미 학습된 모델을 가져와라. 라는 의미입니다.

bert-base-uncased는 Moedl ID입니다.

대략 이런 구조입니다.

Hugging Face Hub
- bert-base-uncased
- gpt2
- klue/roberta-base
- meta-llama


4. 모델과 Tokenizer는 항상 세트라고 생각하자

bert_model = AutoModel.from_pretrained('bert-base-uncased')

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

왜 모델과 Tokenizer를 둘 다 받아야 할까요?

모델은 숫자를 입력으로 받기 때문입니다.

"What is Huggingface Transformers?"

를 모델에 바로 넣을 수 없습니다.

Tokenizer가 "문장 -> 토큰화 -> 숫자"를 수행합니다.

중요한 점은 모델마다 Tokenizer가 다를 수 있다는 것입니다.

따라서 일반적으로

model_id = "어떤 모델"

model = AutoModel.from_pretrained(model_id)

tokenizer = AutoTokenizer.from_pretrained(model_id)

처럼 동일한 model_id를 사용합니다.

5. 중요한 부분

encoded_input = bert_tokenizer(

  text,

  return_tensors = 'pt'

)

Tokenizer가 텍스트를 숫자로 변환합니다.

예를 들어 결과가 개념적으로

{
  "input_ids": tensor([[101, 2054, 2003, ...]]),
  "attention_mask": tensor([[1, 1, 1, ...]])
}

처럼 됩니다.

return_tensors = 'pt'에서 pt = PyTorch 입니다.

즉, "결과를 Python list가 아니라 PyTorch Tensor로 만들어줘." 입니다.

6. encode_input은 무엇인가?

bert_output = bert_model(encoded_input)

처음 보면 굉장히 이상합니다.

encoded_input이

{
  "input_ids": input_ids,
  "attention_mask": attention_mask
}

라고 한다면

bert_model(**encoded_input)

는 사실

bert_model(
  input_ids = input_ids,
  attention_mask=attention_mask
)

와 같습니다. Python의 **는 dictionary를 키워드 인자 형태로 풀어주는 문법입니다.

7. 예제 3.1이 전달하는 진짜 메시지

BERT에서

bert_model = AutoModel.from_pretrained(...)

bert_tokenizer = AutoTokenizer.from_pretrained(...)






# 3.3절 허깅페이스 라이브러리 사용법 익히기

## 예제 3.2. 모델 아이디로 모델 불러오기

In [ ]:
from transformers import AutoModel
model_id = 'klue/roberta-base'
model = AutoModel.from_pretrained(model_id)

## 예제 3.4. 분류 헤드가 포함된 모델 불러오기

In [ ]:
from transformers import AutoModelForSequenceClassification
model_id = 'SamLowe/roberta-base-go_emotions'
classification_model = AutoModelForSequenceClassification.from_pretrained(model_id)

## 예제 3.6. 분류 헤드가 랜덤으로 초기화된 모델 불러오기

In [ ]:
from transformers import AutoModelForSequenceClassification
model_id = 'klue/roberta-base'
classification_model = AutoModelForSequenceClassification.from_pretrained(model_id)

## 예제 3.8. 토크나이저 불러오기

In [ ]:
from transformers import AutoTokenizer
model_id = 'klue/roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_id)

## 예제 3.9. 토크나이저 사용하기

In [ ]:
tokenized = tokenizer("토크나이저는 텍스트를 토큰 단위로 나눈다")
print(tokenized)
# {'input_ids': [0, 9157, 7461, 2190, 2259, 8509, 2138, 1793, 2855, 5385, 2200, 20950, 2],
#  'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
#  'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

print(tokenizer.convert_ids_to_tokens(tokenized['input_ids']))
# ['[CLS]', '토크', '##나이', '##저', '##는', '텍스트', '##를', '토', '##큰', '단위', '##로', '나눈다', '[SEP]']

print(tokenizer.decode(tokenized['input_ids']))
# [CLS] 토크나이저는 텍스트를 토큰 단위로 나눈다 [SEP]

print(tokenizer.decode(tokenized['input_ids'], skip_special_tokens=True))
# 토크나이저는 텍스트를 토큰 단위로 나눈다

## 예제 3.10. 토크나이저에 여러 문장 넣기

In [ ]:
tokenizer(['첫 번째 문장', '두 번째 문장'])

# {'input_ids': [[0, 1656, 1141, 3135, 6265, 2], [0, 864, 1141, 3135, 6265, 2]],
# 'token_type_ids': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]],
# 'attention_mask': [[1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}

## 예제 3.11. 하나의 데이터에 여러 문장이 들어가는 경우

In [ ]:
tokenizer([['첫 번째 문장', '두 번째 문장']])

# {'input_ids': [[0, 1656, 1141, 3135, 6265, 2, 864, 1141, 3135, 6265, 2]],
# 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
# 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

## 예제 3.12. 토큰 아이디를 문자열로 복원

In [ ]:
first_tokenized_result = tokenizer(['첫 번째 문장', '두 번째 문장'])['input_ids']
tokenizer.batch_decode(first_tokenized_result)
# ['[CLS] 첫 번째 문장 [SEP]', '[CLS] 두 번째 문장 [SEP]']

second_tokenized_result = tokenizer([['첫 번째 문장', '두 번째 문장']])['input_ids']
tokenizer.batch_decode(second_tokenized_result)
# ['[CLS] 첫 번째 문장 [SEP] 두 번째 문장 [SEP]']

## 예제 3.13. BERT 토크나이저와 RoBERTa 토크나이저

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained('klue/bert-base')
bert_tokenizer([['첫 번째 문장', '두 번째 문장']])
# {'input_ids': [[2, 1656, 1141, 3135, 6265, 3, 864, 1141, 3135, 6265, 3]],
# 'token_type_ids': [[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]],
# 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

roberta_tokenizer = AutoTokenizer.from_pretrained('klue/roberta-base')
roberta_tokenizer([['첫 번째 문장', '두 번째 문장']])
# {'input_ids': [[0, 1656, 1141, 3135, 6265, 2, 864, 1141, 3135, 6265, 2]],
# 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
# 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

en_roberta_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
en_roberta_tokenizer([['first sentence', 'second sentence']])
# {'input_ids': [[0, 9502, 3645, 2, 2, 10815, 3645, 2]],
# 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1]]}

## 예제 3.14. attention_mask 확인

In [ ]:
tokenizer(['첫 번째 문장은 짧다.', '두 번째 문장은 첫 번째 문장 보다 더 길다.'], padding='longest')

# {'input_ids': [[0, 1656, 1141, 3135, 6265, 2073, 1599, 2062, 18, 2, 1, 1, 1, 1, 1, 1],
# [0, 864, 1141, 3135, 6265, 2073, 1656, 1141, 3135, 6265, 3632, 831, 647, 2062, 18, 2]],
# 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
# [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

## 예제 3.15. KLUE MRC 데이터셋 다운로드

In [ ]:
from datasets import load_dataset
klue_mrc_dataset = load_dataset('klue', 'mrc')
# klue_mrc_dataset_only_train = load_dataset('klue', 'mrc', split='train')

## 예제 3.16. 로컬의 데이터 활용하기
(안내) 아래 코드를 실행하기 위해서는 구글 코랩에 csv 파일이 업로드 되어야 합니다. 허깅페이스 datasets 형식으로 쉽게 변환할 수 있다는 점을 보여주기 위한 예시 코드입니다.

In [ ]:
from datasets import load_dataset
# 로컬의 데이터 파일을 활용
dataset = load_dataset("csv", data_files="my_file.csv")

# 파이썬 딕셔너리 활용
from datasets import Dataset
my_dict = {"a": [1, 2, 3]}
dataset = Dataset.from_dict(my_dict)

# 판다스 데이터프레임 활용
from datasets import Dataset
import pandas as pd
df = pd.DataFrame({"a": [1, 2, 3]})
dataset = Dataset.from_pandas(df)

# 3.4절 모델 학습시키기

## 예제 3.17. 모델 학습에 사용할 연합뉴스 데이터셋 다운로드

In [ ]:
from datasets import load_dataset
klue_tc_train = load_dataset('klue', 'ynat', split='train')
klue_tc_eval = load_dataset('klue', 'ynat', split='validation')
klue_tc_train

In [ ]:
klue_tc_train[0]

In [ ]:
klue_tc_train.features['label'].names
# ['IT과학', '경제', '사회', '생활문화', '세계', '스포츠', '정치']

## 예제 3.18. 실습에 사용하지 않는 불필요한 컬럼 제거

In [ ]:
klue_tc_train = klue_tc_train.remove_columns(['guid', 'url', 'date'])
klue_tc_eval = klue_tc_eval.remove_columns(['guid', 'url', 'date'])
klue_tc_train

## 예제 3.19. 카테고리를 문자로 표기한 label_str 컬럼 추가

In [ ]:
klue_tc_train.features['label']
# ClassLabel(names=['IT과학', '경제', '사회', '생활문화', '세계', '스포츠', '정치'], id=None)

klue_tc_train.features['label'].int2str(1)
# '경제'

klue_tc_label = klue_tc_train.features['label']

def make_str_label(batch):
  batch['label_str'] = klue_tc_label.int2str(batch['label'])
  return batch

klue_tc_train = klue_tc_train.map(make_str_label, batched=True, batch_size=1000)

klue_tc_train[0]
# {'title': '유튜브 내달 2일까지 크리에이터 지원 공간 운영', 'label': 3, 'label_str': '생활문화'}

## 예제 3.20. 학습/검증/테스트 데이터셋 분할

In [ ]:
train_dataset = klue_tc_train.train_test_split(test_size=10000, shuffle=True, seed=42)['test']
dataset = klue_tc_eval.train_test_split(test_size=1000, shuffle=True, seed=42)
test_dataset = dataset['test']
valid_dataset = dataset['train'].train_test_split(test_size=1000, shuffle=True, seed=42)['test']

## 예제 3.21. Trainer를 사용한 학습: (1) 준비

In [ ]:
import torch
import numpy as np
from transformers import (
    Trainer,
    TrainingArguments,
    AutoModelForSequenceClassification,
    AutoTokenizer
)

def tokenize_function(examples):
    return tokenizer(examples["title"], padding="max_length", truncation=True)

model_id = "klue/roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=len(train_dataset.features['label'].names))
tokenizer = AutoTokenizer.from_pretrained(model_id)

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

## 예제 3.22. Trainer를 사용한 학습: (2) 학습 인자와 평가 함수 정의

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    push_to_hub=False
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()}

## 예제 3.23. Trainer를 사용한 학습 - (3) 학습 진행

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

trainer.evaluate(test_dataset) # 정확도 0.84

## 예제 3.24. Trainer를 사용하지 않는 학습: (1) 학습을 위한 모델과 토크나이저 준비

In [ ]:
import torch
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.optim import AdamW

def tokenize_function(examples): # 제목(title) 컬럼에 대한 토큰화
    return tokenizer(examples["title"], padding="max_length", truncation=True)

# 모델과 토크나이저 불러오기
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "klue/roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=len(train_dataset.features['label'].names))
tokenizer = AutoTokenizer.from_pretrained(model_id)
model.to(device)

## 예제 3.25 Trainer를 사용하지 않는 학습: (2) 학습을 위한 데이터 준비

In [ ]:
def make_dataloader(dataset, batch_size, shuffle=True):
    dataset = dataset.map(tokenize_function, batched=True).with_format("torch") # 데이터셋에 토큰화 수행
    dataset = dataset.rename_column("label", "labels") # 컬럼 이름 변경
    dataset = dataset.remove_columns(column_names=['title']) # 불필요한 컬럼 제거
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

# 데이터로더 만들기
train_dataloader = make_dataloader(train_dataset, batch_size=8, shuffle=True)
valid_dataloader = make_dataloader(valid_dataset, batch_size=8, shuffle=False)
test_dataloader = make_dataloader(test_dataset, batch_size=8, shuffle=False)

## 예제 3.26. Trainer를 사용하지 않는 학습: (3) 학습을 위한 함수 정의

In [ ]:
def train_epoch(model, data_loader, optimizer):
    model.train()
    total_loss = 0
    for batch in tqdm(data_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device) # 모델에 입력할 토큰 아이디
        attention_mask = batch['attention_mask'].to(device) # 모델에 입력할 어텐션 마스크
        labels = batch['labels'].to(device) # 모델에 입력할 레이블
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels) # 모델 계산
        loss = outputs.loss # 손실
        loss.backward() # 역전파
        optimizer.step() # 모델 업데이트
        total_loss += loss.item()
    avg_loss = total_loss / len(data_loader)
    return avg_loss

## 예제 3.27. Trainer를 사용하지 않는 학습: (4) 평가를 위한 함수 정의

In [ ]:
def evaluate(model, data_loader):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    with torch.no_grad():
        for batch in tqdm(data_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            logits = outputs.logits
            loss = outputs.loss
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(data_loader)
    accuracy = np.mean(np.asarray(predictions) == np.asarray(true_labels))
    return avg_loss, accuracy

## 예제 3.28 Trainer를 사용하지 않는 학습: (5) 학습 수행

In [ ]:
num_epochs = 1
optimizer = AdamW(model.parameters(), lr=5e-5)

# 학습 루프
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    train_loss = train_epoch(model, train_dataloader, optimizer)
    print(f"Training loss: {train_loss}")
    valid_loss, valid_accuracy = evaluate(model, valid_dataloader)
    print(f"Validation loss: {valid_loss}")
    print(f"Validation accuracy: {valid_accuracy}")

# Testing
_, test_accuracy = evaluate(model, test_dataloader)
print(f"Test accuracy: {test_accuracy}") # 정확도 0.82

## 예제 3.29. 허깅페이스 허브에 모델 업로드

In [ ]:
# 모델의 예측 아이디와 문자열 레이블을 연결할 데이터를 모델 config에 저장
id2label = {i: label for i, label in enumerate(train_dataset.features['label'].names)}
label2id = {label: i for i, label in id2label.items()}
model.config.id2label = id2label
model.config.label2id = label2id

In [ ]:
from huggingface_hub import login

login(token="본인의 허깅페이스 토큰 입력")
repo_id = f"본인의 아이디 입력/roberta-base-klue-ynat-classification"
# Trainer를 사용한 경우
trainer.push_to_hub(repo_id)
# 직접 학습한 경우
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

# 3.5절 모델 추론하기

## 예제 3.30. 학습한 모델을 불러와 pipeline을 활용해 추론하기

In [ ]:
# 실습을 새롭게 시작하는 경우 데이터셋 다시 불러오기 실행
# import torch
# import torch.nn.functional as F
# from datasets import load_dataset

# dataset = load_dataset("klue", "ynat", split="validation")

In [ ]:
from transformers import pipeline

model_id = "본인의 아이디 입력/roberta-base-klue-ynat-classification"

model_pipeline = pipeline("text-classification", model=model_id)

model_pipeline(dataset["title"][:5])

## 예제 3.31. 커스텀 파이프라인 구현

In [ ]:
import torch
from torch.nn.functional import softmax
from transformers import AutoModelForSequenceClassification, AutoTokenizer

class CustomPipeline:
    def __init__(self, model_id):
        self.model = AutoModelForSequenceClassification.from_pretrained(model_id)
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model.eval()

    def __call__(self, texts):
        tokenized = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True)

        with torch.no_grad():
            outputs = self.model(**tokenized)
            logits = outputs.logits

        probabilities = softmax(logits, dim=-1)
        scores, labels = torch.max(probabilities, dim=-1)
        labels_str = [self.model.config.id2label[label_idx] for label_idx in labels.tolist()]

        return [{"label": label, "score": score.item()} for label, score in zip(labels_str, scores)]

custom_pipeline = CustomPipeline(model_id)
custom_pipeline(dataset['title'][:5])